## 2. DETECÇÃO DE FRAUDE EM TRANSAÇÕES FINANCEIRAS

- **Problema:** classificar transações sintéticas como fraudulentas/legítimas, simulando o desafio de detecção em sistemas como o PIX (desbalanceamento extremo).
- **Dataset:** PaySim Synthetic Financial — Kaggle — CC BY-SA 4.0 — ~470 MB, 6,3 M de transações.
- **Abordagem:** XGBoost / Random Forest com SMOTE/class weights vs. baseline por threshold. Meta: F1 ≥ 0.85 na classe minoritária.
- **Áreas exercitadas:** Ciência de Dados / Machine Learning.
- **Diferencial:** matriz de custo configurável via slider em dashboard Streamlit.
- **Tendência 2026:** após o MED 2.0 do BCB (fev/2026), modelos de score de fraude tornaram-se obrigatórios em pagamentos instantâneos.
- **Dificuldade:** *

In [ ]:
# Imports que serão usados no decorrer do código

import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from sklearn.metrics import classification_report

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

# Configurações visuais globais
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

#Entendimento Inicial

### Análise

O dataset possui **6.362.620 registros** e **11 variáveis**, indicando um volume grande de dados, adequado para problemas de detecção de fraude.

Em relação aos tipos de dados:
- Variáveis numéricas: `step`, `amount`, saldos e variáveis alvo  
- Variáveis categóricas: `type`, `nameOrig`, `nameDest`  

Não há indícios de valores nulos, já que o `count` é igual para todas as colunas.

---

### Insights

- A variável `amount` apresenta **alta variabilidade** (std elevado), indicando presença de valores extremos  
- Os saldos (`oldbalance` e `newbalance`) possuem muitos valores iguais a **zero**, especialmente nos quartis inferiores  
- O valor máximo das transações é muito elevado, indicando presença de **outliers**  
- A média de `isFraud` confirma que fraudes são **extremamente raras**  
- `isFlaggedFraud` tem média praticamente zero → quase nunca é ativado  

---

### Observações

- Os IDs (`nameOrig`, `nameDest`) são categóricos de **alta cardinalidade** e não podem ser usados diretamente em modelos sem tratamento  
- As variáveis de saldo (`oldbalance`, `newbalance`) podem causar **data leakage** se usadas diretamente na modelagem  
- A grande presença de zeros nos saldos pode indicar:
  - contas sem movimentação  
  - ausência de informação (especialmente para merchants)  
- A alta dispersão nos dados pode impactar modelos → pode ser necessário normalização ou transformação (log)  
- O dataset é **sintético**, podendo não refletir perfeitamente o comportamento do mundo real  

---

### Conclusão

O dataset apresenta boa integridade estrutural (sem valores nulos), porém possui alta variabilidade, presença significativa de outliers e muitos valores zerados em variáveis de saldo. Além disso, confirma-se o forte desbalanceamento da variável alvo, exigindo cuidados específicos na modelagem e avaliação dos modelos.

In [ ]:
files = os.listdir(path)
print(files)

df = pd.read_csv(os.path.join(path, files[0]))
df.head()

In [ ]:
df.info()
df.describe()

#EDA (Análise Exploratória)

### Análise

Os tipos de transação mostram que `CASH_OUT` e `PAYMENT` são mais frequentes, enquanto `TRANSFER` e `DEBIT` possuem menor volume.

Ao cruzar com `isFraud`, observa-se que:
- Fraudes ocorrem **apenas em `CASH_OUT` e `TRANSFER`**
- `TRANSFER` possui a maior taxa de fraude  
- `CASH_OUT` tem menor taxa, mas alto volume absoluto  

A variável `amount` apresenta:
- Forte **assimetria à direita** (média > mediana)  
- Alta dispersão e presença de **outliers** (até ~92 milhões)  
- Concentração de valores baixos com cauda longa  

O boxplot (escala log) mostra que:
- `TRANSFER` e `CASH_OUT` concentram os maiores valores  
- `PAYMENT` e `DEBIT` possuem valores menores  

---

### Insights

- Fraudes estão concentradas em tipos específicos e de maior valor  
- `TRANSFER` é o mais arriscado proporcionalmente  
- `CASH_OUT` combina **alto volume + fraude**  
- Transações com valores mais altos tendem a estar associadas a fraude  
- A distribuição de `amount` reforça padrões importantes para detecção  

---

### Observações

- O padrão de fraude é muito claro → pode facilitar demais o modelo (dado simulado)  
- A variável `type` será extremamente relevante, com risco de dependência excessiva  
- A presença de outliers exige tratamento (ex: log1p)  
- Modelos sensíveis à escala podem ser impactados  
- Valores zero devem ser investigados  

---

### Conclusão

Os resultados mostram que fraude está fortemente associada a tipos específicos (`TRANSFER` e `CASH_OUT`) e a transações de maior valor. As variáveis `type` e `amount` são altamente preditivas, mas exigem cuidado devido à distribuição assimétrica, presença de outliers e possível simplificação do problema por padrões muito evidentes.


In [ ]:
# Distribuição da variável type
df['type'].value_counts()

In [ ]:
# Distribuição da variável alvo
fraud_rate = df["isFraud"].value_counts(normalize=True)
print(fraud_rate)
print(f"\nApenas {fraud_rate[1]:.4%} das transações são fraude, o dataset é altamente desbalanceado")

In [ ]:
# Tipos de transação vs fraude
pd.crosstab(df['type'], df['isFraud'], normalize='index')

In [ ]:
# Distribuição de valores (amount)
df['amount'].describe()

In [ ]:
# Distribuição de valores (amount) em gráfico
# Escala linear (esquerda) e log (direita)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df["amount"].hist(bins=50, ax=axes[0])
axes[0].set_title("Distribuição linear de amount")
axes[0].set_xlabel("Valor da transação")
axes[0].set_ylabel("Frequência")

df["amount"].apply(np.log1p).hist(bins=50, ax=axes[1], color="steelblue")
axes[1].set_title("Distribuição log(amount + 1)")
axes[1].set_xlabel("log(valor + 1)")
axes[1].set_ylabel("Frequência")

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot de amount por tipo de transação

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="type", y="amount", hue="type", palette="Blues", legend=False)
plt.yscale("log")
plt.title("Distribuição de valores por tipo de transação")
plt.xlabel("Tipo de transação")
plt.ylabel("Valor (escala log)")
plt.tight_layout()
plt.show()

In [ ]:
# Análise temporal
df['step'].describe()

In [ ]:
fraud_by_step = df.groupby("step")["isFraud"].mean()

fraud_by_step.plot(figsize=(10, 4))
plt.title("Taxa de fraude ao longo do tempo (em horas)")
plt.xlabel("Hora da simulação")
plt.ylabel("Taxa de fraude")
plt.tight_layout()
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10, 6))
sns.heatmap(corr, annot=True,  fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5, annot_kws={"size": 8})
plt.title("Matriz de correlação")
plt.tight_layout()
plt.show()

A análise de correlação indica que não há relações lineares fortes com a variável de fraude, reforçando a necessidade de modelos mais robustos. Além disso, evidencia redundância nas variáveis de saldo, que devem ser tratadas com cautela na modelagem.

#Qualidade dos Dados (Data Quality)

### Análise

O dataset não apresenta valores nulos nem registros duplicados, indicando boa integridade estrutural inicial.

Entretanto, ao verificar consistência lógica:
- Há **5.413.997 inconsistências** na relação  
  `oldbalanceOrg - amount ≠ newbalanceOrig`  
- Isso indica que grande parte dos dados não segue a regra básica de atualização de saldo  

Na variável `isFlaggedFraud`:
- Apenas **16 transações foram sinalizadas** como fraude  
- Dessas, todas são realmente fraude  
- Porém, existem **8.197 fraudes que NÃO foram sinalizadas**

---

### Insights

- A ausência de nulos e duplicados sugere dataset bem estruturado  
- As inconsistências de saldo são muito frequentes → possível padrão do simulador ou comportamento de fraude  
- O sistema de detecção (`isFlaggedFraud`) tem:
  - **alta precisão** (quando marca, acerta)
  - **baixíssimo recall** (deixa passar quase todas fraudes)

---

### Observações

- As inconsistências podem estar ligadas a:
  - transações intermediárias
  - regras do simulador
  - comportamento fraudulento  
- `isFlaggedFraud` não é confiável como detector → regra muito limitada  
- Pode haver **data leakage** se variáveis de saldo forem usadas diretamente  

---

### Conclusão

Apesar da boa qualidade estrutural (sem nulos ou duplicados), o dataset apresenta inconsistências relevantes nas variáveis de saldo e um sistema de detecção de fraude extremamente limitado. Isso reforça a necessidade de uma modelagem mais robusta para capturar padrões de fraude de forma eficaz.

In [ ]:
# Verificação de valores nulos
nulls = df.isnull().sum()
print(nulls)
print(nulls[nulls > 0] if nulls.any() else "Nenhum valor nulo encontrado")

In [ ]:
# Verificação de duplicatas
dups = df.duplicated().sum()
print(dups)
print(f"Duplicatas: {dups:,} ({dups/len(df):.4%} do total)")

In [ ]:
# Inconsistências lógicas
tolerancia = 1e-2 # Tolerância pra evitar falsos positivos
inconsistencias = (df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]).abs() > tolerancia

print(f"Inconsistências: {inconsistencias.sum():,} ({inconsistencias.mean():.1%} do total)\n")
print("Taxa de inconsistência por classe:")
print(
    df.assign(inconsistente=inconsistencias)
      .groupby("isFraud")["inconsistente"]
      .mean()
      .rename(index={0: "Legítima", 1: "Fraude"})
)

In [ ]:
print(classification_report(df["isFraud"], df["isFlaggedFraud"],
                             target_names=["Legítima", "Fraude"]))

# Pré-processamento

- Encoding de variáveis categóricas (`type`)
- Remoção de colunas irrelevantes (`nameOrig`, `nameDest`, `isFlaggedFraud`)
- Tratamento de outliers em `amount` com log1p
- Separação treino/teste com estratificação por `isFraud`

# Baseline

- Regra simples por limiar de valor (ex: `amount > X` → fraude)
- Comparar precision, recall e F1 com os modelos futuros

# Modelagem

- XGBoost e Random Forest
- Técnicas de balanceamento: SMOTE ou `class_weight="balanced"`
- Meta: F1 ≥ 0.85 na classe minoritária

# Avaliação

- `classification_report` completo
- Matriz de confusão
- Curva ROC e Precision-Recall
- Foco no recall da classe fraude — falsos negativos têm custo alto

# Dashboard (Streamlit)

- Matriz de custo configurável via slider
- Simular impacto financeiro de falsos positivos vs falsos negativos